In [1]:
## Lijst met test cases
import json
try:
    with open('testcases.json', 'r', encoding='utf-8') as f:
        test_cases = json.load(f)

    print(f"Successfully loaded {len(test_cases)} records from file.")
    print("\nFirst record from file:")
    print(test_cases[0])

except FileNotFoundError:
    print("Error: 'data.json' not found. Please ensure the file exists in the current directory.")
except json.JSONDecodeError:
    print("Error: The file content is not valid JSON.")

Successfully loaded 13 records from file.

First record from file:
{'naam': 'Vaarwegmarkeringen Nederland', 'type': 'Dataset (valide)', 'url': 'https://ngr.acceptatie.nationaalgeoregister.nl/geonetwork/srv/api/records/be1b1514-8d1f-48e1-9624-fee9b784138b/formatters/dcat-ap-nl-3?output=xml', 'focus_node': 'https://ngr.acceptatie.nationaalgeoregister.nl/geonetwork/srv/api/records/be1b1514-8d1f-48e1-9624-fee9b784138b#resource'}


In [2]:
## Dit is de standaard pyshacl validatie functie

from pyshacl import validate
def do_validate(data_graph, sg, focus_nodes=None):
    r = validate(focus_nodes=focus_nodes,
      shacl_graph=sg,
      data_graph=data_graph,
      ont_graph=None,
      inference='rdfs',
      abort_on_first=False,
      allow_infos=False,
      allow_warnings=False,
      meta_shacl=False,
      advanced=False,
      js=False,
      debug=False)
    conforms, results_graph, results_text = r
    return conforms, results_graph, results_text

In [3]:
import os
from dotenv import load_dotenv
load_dotenv() 
shacl_folder = os.environ['SHACL_FOLDER']
ttl_folder = os.environ['TTL_FOLDER']
results_folder = os.environ['RESULTS_FOLDER']

In [4]:


from rdflib import Graph
# laad de SHACL shapes

# dcat_ap_minimal =  Graph().parse('https://semiceu.github.io/DCAT-AP/releases/3.0.0/html/shacl/shapes.ttl', format="ttl")

dcat_ap = Graph().parse(shacl_folder + 'dcat-ap-SHACL.ttl', format="ttl")
dcat_ap_nl = Graph().parse(shacl_folder + 'dcat-ap-nl-SHACL.ttl', format="ttl")
klassebereik = Graph().parse(shacl_folder + 'dcat-ap-nl-SHACL-klassebereik.ttl', format="ttl")
codelijsten = Graph().parse(shacl_folder + 'dcat-ap-nl-SHACL-klassebereik-codelijsten.ttl', format="ttl")
aanbevolen = Graph().parse(shacl_folder + 'dcat-ap-nl-SHACL-aanbevolen.ttl', format="ttl")

level_1 = Graph()
# level_1 += dcat_ap_minimal
level_1 += dcat_ap
level_1 += dcat_ap_nl

level_2 = Graph()
level_2 += level_1
level_2 += klassebereik

level_3 = Graph()
level_3 += level_2
level_3 += codelijsten

level_4 = Graph()
level_4 += level_3
level_4 += aanbevolen

In [ ]:
import json
import requests
import os
import re

def clean_filename(name):
    """Sanitizes a string to be a safe filename."""
    # Replace spaces and special characters with underscores
    name = re.sub(r'[^\w\-_\. ]', '_', name)
    # Remove leading/trailing whitespace
    name = name.strip()
    # Replace multiple spaces/underscores with a single hyphen
    name = re.sub(r'[ _\-]+', '-', name)
    return name.lower()

def download_xml_file(data):
    """
    Iterates through the list of data dictionaries, fetches the XML from the 
    URL, and saves it to a file.
    """
    download_dir = "xml"
    os.makedirs(download_dir, exist_ok=True)
    # print(f"Downloads will be saved in the '{download_dir}/' directory.")

    download_count = 0
    
    for item in data:
        naam = item.get("naam", "unknown_name")
        item_type = item.get("type", "unknown_type").replace(" ", "_").replace("(", "").replace(")", "").lower()
        url = item.get("url")
        
        if not url:
            print(f"Skipping record for '{naam}' - No URL provided.")
            continue
            
        # Create a unique filename: clean_name_type_index.xml
        base_name = clean_filename(f"{naam}_{item_type}")
        
        # Check if the URL has an ID we can use for better uniqueness
        # This extracts 'be1b1514-8d1f-48e1-9624-fee9b784138b' from the URL
        match = re.search(r'records/([\w\-]+)', url)
        record_id = match.group(1) if match else None
        
        # Construct the final filename
        if record_id:
            filename = os.path.join(download_dir, f"{base_name}_{record_id}.xml")
        else:
            download_count += 1
            filename = os.path.join(download_dir, f"{base_name}_{download_count}.xml")

        # Skip if file already exists to prevent re-downloading
        if os.path.exists(filename):
            print(f"Skipping: File already exists at {filename}")
            continue
            
        try:
            # Send an HTTP GET request to the URL
            response = requests.get(url, timeout=15)
            response.raise_for_status() # Raise exception for bad status codes (4xx or 5xx)
            
            # Save the content to the local file
            with open(filename, 'wb') as f:
                f.write(response.content)
            
            download_count += 1
            # print(f"SUCCESS: Saved '{naam}' ({item_type}) to {filename}")
            return filename
        
        except requests.exceptions.RequestException as e:
            print(f"FAILURE: Could not download {url} for '{naam}'. Error: {e}")
        except IOError as e:
            print(f"FAILURE: Could not write file {filename}. Error: {e}")


    print(f"\n--- Download complete. Successfully processed {download_count} items. ---")
    

In [33]:
#lees rdfxml en converteer naar ttl => een test of de rdfxml valide is en ttl is makkelijker leesbaar dan rdfxml
from rdflib import Graph
from rdflib.exceptions import Error

def rdfxml_2_ttl(rdf_file, ttl_file):
    try:
        graph = Graph()
        graph.parse(rdf_file)
    
        graph.serialize(ttl_file, format='turtle')
    except Error as err:
        # print('Error during rdf to ttl transformation', err)
        return err

In [ ]:
from pyshacl.errors import ReportableRuntimeError

def validate_ttl(ttl_file, level, focus_node):
    
    try:
        data_graph = Graph()
        data_graph.parse(ttl_file, format='turtle')
        # sg = Graph()
        # sg.parse(level_1,  format='turtle')

        sh_out = results_folder + 'shacl\\' + os.path.basename(ttl_file).replace('.ttl','.shacl.ttl')
        txt_out = results_folder + 'report\\' + os.path.basename(ttl_file).replace('.ttl','.txt')

        conforms, results_graph, results_text = do_validate(data_graph, level, focus_node)
        results_graph.serialize(sh_out)
        f = open(txt_out, "a")
        f.write(results_text)
        f.close()
        # print(sh_out + ' Validation result: ' + str(conforms))
    except ReportableRuntimeError as err:
        print('Error during shacl validation', err)

In [41]:
rdf_serialisationError = {}

for case in test_cases:
   filename = download_xml_file([case])
   try:
      ttl_out = ttl_folder + os.path.basename(filename).replace('xml','ttl')
   except Exception as e:
      continue
   res = rdfxml_2_ttl(filename, ttl_out)
   print(res)
   if res is not None:
      rdf_serialisationError[filename] = res
   else:
      try:
         validate_ttl(ttl_file=ttl_out,level=level_1, focus_node=case['focus_node'])
      except Exception as e:
         print(f"Validation failed for {ttl_out} with error: {e}")

Downloads will be saved in the 'xml/' directory.
file:///C:/niels/1-Projects/GeoDCAT-AP/ISO-2-DCAT/dcat-ap-nl-3/Valideer_dcat_ap_nl3/xml/vaarwegmarkeringen-nederland-dataset-valide_be1b1514-8d1f-48e1-9624-fee9b784138b.xml:179:6: Invalid property attribute URI: http://www.w3.org/1999/02/22-rdf-syntax-ns#about
Downloads will be saved in the 'xml/' directory.
None
C:\\niels\\1-Projects\\GeoDCAT-AP\\ISO-2-DCAT\\dcat-ap-nl-3\\Valideer_dcat_ap_nl3\\results\\shacl\vaarwegmarkeringen-nederland-service-atom-valide_252db472-201d-430b-9c55-4d37563787bc.shacl.ttl Validation result: True
Downloads will be saved in the 'xml/' directory.
None
C:\\niels\\1-Projects\\GeoDCAT-AP\\ISO-2-DCAT\\dcat-ap-nl-3\\Valideer_dcat_ap_nl3\\results\\shacl\vaarwegmarkeringen-nederland-service-wfs-valide_82677435-ce93-4a79-94dd-bbc87fb61f36.shacl.ttl Validation result: True
Downloads will be saved in the 'xml/' directory.
None
C:\\niels\\1-Projects\\GeoDCAT-AP\\ISO-2-DCAT\\dcat-ap-nl-3\\Valideer_dcat_ap_nl3\\results\\s

In [10]:
import pandas as pd

def create_results(ttl_file):
    data_graph = Graph()
    data_graph.parse(ttl_file, format='turtle')
    # Define your SPARQL query
    sparql_query = """
    PREFIX sh: <http://www.w3.org/ns/shacl#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX dcat: <http://www.w3.org/ns/dcat#>
    PREFIX dcterms: <http://purl.org/dc/terms/>

        # This query summarizes the validation results by grouping them
        # by the violated property and the constraint component.
        SELECT
            # (COUNT(DISTINCT ?result) AS ?TotalErrors)
            # (STR(?resultPath) AS ?ViolatedPropertyURI)
            (STR(?focusNode) AS ?nodeName)
            (STR(REPLACE(STR(?resultPath), ".*/", "")) AS ?PropertyName)
            (STR(?sourceShape) AS ?SourceShape)
            (STR(REPLACE(STR(?sourceComponent), ".*#", "")) AS ?ConstraintComponentURI)
            #(STR(REPLACE(STR(?sourceComponent), ".*/", "")) AS ?ConstraintName)
            ?resultMessage
        WHERE {
            # Find the validation report
            ?report a sh:ValidationReport .
            
            # Find all validation results linked to the report
            ?report sh:result ?result .
            
            # Extract the key details for each result
            ?result sh:focusNode ?focusNode .
            ?result sh:resultPath ?resultPath .
            ?result sh:sourceShape ?sourceShape .
            ?result sh:sourceConstraintComponent ?sourceComponent .
            ?result sh:resultMessage ?resultMessage .
            ?result sh:resultSeverity sh:Violation . # Only include Violations, ignore Warnings/Infos
        }
        # Group by the specific violation details (Property, Constraint, Message)
        GROUP BY ?nodeName ?resultPath ?sourceComponent ?resultMessage
        ORDER BY ?nodeName ?resultPath
        """
    # Execute the query
    res = data_graph.query(sparql_query)
    
    return  res


In [11]:
def sanitize_message(message):
    # This regex finds the focus node URI (between < and >)
    # and replaces it with a simple placeholder or extracts the record ID
    record_id_match = re.search(r'records/([\w-]+)#resource', message)
    if record_id_match:
        short_id = f"Record ID: {record_id_match.group(1)[:8]}..."
        # Replace the entire URI (including the <> brackets)
        return re.sub(r'<[^>]*#resource>', short_id, message)
    return message

In [48]:
from datetime import datetime
from pathlib import Path

f = open("./resultaat.md", "w", encoding='utf-8')
f.write("# SHACL Validatie DCAT-AP-NL Bestanden\n\n")
f.write("\n\n")
f.write("Gegenereerd op: " + datetime.now().strftime("%Y-%m-%d %H:%M:%S") + "\n\n")

f.write("## RDF Serialisatie Fouten\n\n")

for k, v in rdf_serialisationError.items():
    f.write(str(v).split("/xml/")[-1] + "\n\n")

f.write("\n\n")

f.write("## SHACL Validatie Resultaten\n\n")

shacl_results = Path(results_folder + 'shacl\\')


for ttl_file in shacl_results.glob('*.ttl'):
    print(ttl_file)
    f.write("File: " + str(ttl_file).split("\\")[-1] + "\n\n")
    try:
        
        results = create_results(ttl_file)

        results_list = [dict(zip(results.vars, row)) for row in results]
        df = pd.DataFrame(results_list)

        if df.empty:    
            f.write("Geen validatiefouten gevonden \n\n")
        else:
            # Trimming the column names
            df.columns = df.columns.str.strip()        
            df['resultMessage'] = df['resultMessage'].apply(sanitize_message)
            f.write(df.to_markdown(index=False))
            f.write("\n\n") 
    except ReportableRuntimeError as err:
        print('Error during query', err)

f.close()

C:\niels\1-Projects\GeoDCAT-AP\ISO-2-DCAT\dcat-ap-nl-3\Valideer_dcat_ap_nl3\results\shacl\bathymetrie-nederland-binnenwateren-1-mtr.-serie-landelijke-opslagsyssteem-lodingen-lol-service-wcs-valide_9d9573b0-bd3b-4d7b-a97d-e65841e6e0b2.shacl.ttl
C:\niels\1-Projects\GeoDCAT-AP\ISO-2-DCAT\dcat-ap-nl-3\Valideer_dcat_ap_nl3\results\shacl\bathymetrie-nederland-binnenwateren-1-mtr.-serie-landelijke-opslagsyssteem-lodingen-lol-service-wms-valide_9d9573b0-bd3b-4d7b-a97d-e65841e6e0b4.shacl.ttl
C:\niels\1-Projects\GeoDCAT-AP\ISO-2-DCAT\dcat-ap-nl-3\Valideer_dcat_ap_nl3\results\shacl\cbs-vierkantstatistieken-100m-service-100m-2000-wfs-valide_e79ffdbb-f5cf-44ce-b5c2-ff4cbdf0abf4.shacl.ttl
C:\niels\1-Projects\GeoDCAT-AP\ISO-2-DCAT\dcat-ap-nl-3\Valideer_dcat_ap_nl3\results\shacl\cbs-vierkantstatistieken-100m-service-100m-2000-wms-valide_602ad34d-fe51-48f4-8224-95835144cf68.shacl.ttl
C:\niels\1-Projects\GeoDCAT-AP\ISO-2-DCAT\dcat-ap-nl-3\Valideer_dcat_ap_nl3\results\shacl\regionale-wandelnetwerken-data

In [44]:
ttl_file = r'C:\\niels\\1-Projects\\GeoDCAT-AP\\ISO-2-DCAT\\dcat-ap-nl-3\\Valideer_dcat_ap_nl3\\results\\shacl\\vaarwegmarkeringen-nederland-service-atom-valide_252db472-201d-430b-9c55-4d37563787bc.shacl.ttl'

results = create_results(ttl_file)

results_list = [dict(zip(results.vars, row)) for row in results]
df = pd.DataFrame(results_list)

df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 0 entries
Empty DataFrame
